# sqrt-eps-stabilize — worked example 2: Layer normalization with eps-stabilized denominator

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sqrt-eps-stabilize`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Layer normalization normalizes across the feature dimension for each sample independently: `(x - mean) / sqrt(var + eps)`. The eps must go inside the sqrt for two reasons: it prevents divide-by-zero when all features of a sample are identical, and it keeps the backward-pass gradient well-behaved near zero variance. This is the standard pattern used in transformer architectures.

## Worked solution

**Step 1 — Create a (3, 4) batch.** Three samples, four features. We artificially make the second sample have all-equal features (zero variance).

**Step 2 — Compute mean and variance per sample.** `x.mean(dim=-1, keepdim=True)` and `x.var(dim=-1, unbiased=False, keepdim=True)` give shape (3, 1).

**Step 3 — Normalize with eps inside sqrt.** The keepdim=True is critical — it enables broadcasting subtraction and division against (3, 4).

**Step 4 — Verify.** The normalized samples should have zero mean and unit variance (except for the zero-variance sample, which comes out as all-zeros, which is correct and finite).

In [ ]:
import torch as t

t.manual_seed(42)

# (3, 4) batch; second sample has zero variance
x = t.tensor([
    [1.0, 3.0, 2.0, 4.0],
    [5.0, 5.0, 5.0, 5.0],  # zero variance
    [0.0, -1.0, 2.0, 1.0],
], dtype=t.float32)

eps = 1e-5
mean = x.mean(dim=-1, keepdim=True)           # (3, 1)
var  = x.var(dim=-1, unbiased=False, keepdim=True)  # (3, 1)
normed = (x - mean) / t.sqrt(var + eps)      # (3, 4)

print('var of each sample:', var.squeeze().tolist())
print('Normalized:\n', normed)
print('All finite:', t.isfinite(normed).all().item())
print('Sample 0 mean (near 0):', normed[0].mean().item())
print('Sample 1 all zeros (const input):', t.allclose(normed[1], t.zeros(4), atol=1e-4))